Author: Robin Sternberg  
This notebook is licensed under Creative Commons Attribution-ShareAlike 4.0

# Daten der Abgeordneten

Die Stammdaten aller bisherigen Abgeordneten im Bundestag gibt es [hier](https://www.bundestag.de/services/opendata) unter "Weitere Informationen" zum Download.  
In diesem Notebook werden die Daten ausgelesen, mit ein paar zusätzlichen Einträgen angereichert und dann zur weiteren Benutzung im Projekt im CSV-Format abgespeichert.

\*Anmerkung: "MDB" steht als Abkürzung für Mitglied des Bundestags

In [1]:
import xml.etree.ElementTree as ET

Der folgende Codeblock liest die Stammdaten-XML-Datei ein und macht daraus eine Liste von Tupeln, die die Namen, Parteien und Wahlperioden von Mitgliedern des Bundestages enthalten.  
Dabei werden alle Namen eines Mitglieds durchgegangen und es wird überprüft, ob der volle Name bereits verarbeitet wurde.  
Namen mit Abkürzungen werden bei älteren Wahlperioden (vor der 18. Wahlperiode) ergänzt, um beide Varianten (mit und ohne Abkürzung) zu erfassen. Ab der 18. Wahlperiode sind unterschiedliche Anreden schon in der XML-Datei enthalten.

In [2]:
tree = ET.parse('../_data/MDB_STAMMDATEN.XML')
root = tree.getroot()
bt_list = []

for mdb in root.findall('MDB'):
    processed_names = set()
    
    # always get the latest election period in which the MDB was in the BT
    latest_wp_element = mdb.find('.//WAHLPERIODE[last()]')
    if latest_wp_element is not None:
        election_period = latest_wp_element.find('WP').text

     # names can change due to marriage etc. -> iterate over all listed names for the MDB
    for name_element in mdb.findall('.//NAME'):
        last_name = name_element.find('NACHNAME').text
        first_name = name_element.find('VORNAME').text
        full_name = f"{first_name} {last_name}"
        
        # we are only want to get multiple entries where first or last name changed, not e.g. Title "Dr." etc.
        if full_name not in processed_names:

            '''
                Edge case, Abbreviated second first names:
                    e.g. Lutz G. Stavenhagen is sometimes referred to by Lutz Stavenhagen in the protocols.
                    This is why we need to put both version "Lutz Stavenhagen" and "Lutz G. Stavenhagen" into the list.

                Edge case of the edge case: Since BT-Period 18, sometimes the data in the xml file reflects if a 
                    MDB changed how he is referred to in the protocols. This would lead to double entries for Albert (H.) Weiler and Tobias (B.) Bacherle.
                    This is why we have to check if the Bundestagsperiode is smaller than 18 before adding the abbreviated part to the list 
                    since beginning with bt-period 18, the "double entry" will automatically be created through the findall('.//NAME')
            '''
            if first_name[-1] == '.' and int(election_period) <18:
                dict_entry_case_abbreviated = {
                'last_name': last_name,
                'first_name': first_name,
                'party': mdb.find('.//PARTEI_KURZ').text,
                'election_period': election_period
                }
                bt_list.append(dict_entry_case_abbreviated)
                first_name = first_name[:-3] # get rid of abbreviation part

            dict_entry = {
                'last_name': last_name,
                'first_name': first_name,
                'party': mdb.find('.//PARTEI_KURZ').text,
                'election_period': election_period
            }
            if dict_entry['party'] in ['BÜNDNIS 90/DIE GRÜNEN', 'DIE GRÜNEN/BÜNDNIS 90']:
                dict_entry['party'] = "GRÜNE"
            
            bt_list.append(dict_entry)
            processed_names.add(full_name)
            
bt_tuples = [(f"{adbt['first_name']} {adbt['last_name']}", adbt['party'], adbt['election_period']) for adbt in bt_list]
print(f"Anzahl von Einträgen: {len(bt_tuples)}")
bt_tuples[:20]

Anzahl von Einträgen: 4492


[('Manfred Abelein', 'CDU', '11'),
 ('Ernst Achenbach', 'FDP', '7'),
 ('Annemarie Ackermann', 'CDU', '4'),
 ('Else Ackermann', 'CDU', '12'),
 ('Ulrich Adam', 'CDU', '16'),
 ('Rudolf Adams', 'SPD', '8'),
 ('Raban Adelmann', 'CDU', '3'),
 ('Konrad Adenauer', 'CDU', '5'),
 ('Brigitte Adler', 'SPD', '14'),
 ('Eduard Adorno', 'CDU', '6'),
 ('Jochen Aerssen', 'CDU', '9'),
 ('Willi Agatz', 'KPD', '1'),
 ('Conrad Ahlers', 'SPD', '8'),
 ('Adolf Ahrens', 'DP', '1'),
 ('Hermann Ahrens', 'SPD', '5'),
 ('Karl Ahrens', 'SPD', '11'),
 ('Heinrich Aigner', 'CSU', '8'),
 ('Siegbert Alber', 'CDU', '8'),
 ('Johannes Albers', 'CDU', '2'),
 ('Luise Albertz', 'SPD', '5')]

In [3]:
for entry in bt_tuples:
    if 'Angela Merkel' in entry[0]:
        print(entry)

('Angela Merkel', 'CDU', '19')


## Hinzufügen von Rednern, die nicht Teil des Bundestags sind.

Im nächsten Notebook wird die hier erstellte Liste genutzt, um für Redner die Partei nachzuschlagen, wenn diese nicht im Text mit vermerkt ist. Das ist z.B. bei amtierenden Bundesministern oder Staatssekretären der Fall.  
Einige Staatssekretäre und Minister waren aber vor ihrem Amt nicht im Bundestag, kommen also auch in der Liste nicht vor. 
Die Wichtigsten (am häufigsten vorkommenden) habe ich noch händisch über Wikipedia herausgesucht. So können deren Reden auch den richtigen Parteien zugeordnet werden.  
An die Liste der wichtigsten fehlenden Redner bin ich dadurch gekommen, dass ich mir beim Parsen (nächstes Notebook) alle Redner ausgegeben habe, bei denen keine Partei gefunden werden konnte.  

In [4]:
bt_tuples.append(("Boris Pistorius", "SPD", '20'))
bt_tuples.append(("Nancy Faeser", "SPD", '20'))
bt_tuples.append(("Klaus-Dieter Fritsche", "CDU/CSU", '19'))
bt_tuples.append(("Aydan Özoguz", "SPD", '19')) # Member of BT but the ğ accent is not always present in the protocols, so we append her manually
bt_tuples.append(("Johanna Wanka", "CDU/CSU", '18'))
bt_tuples.append(("Philipp Rösler", "FDP", '17'))
bt_tuples.append(("Hans-Jürgen Beerfeltz", "FDP", '17'))
bt_tuples.append(("Erich Stather", "SPD",'16'))
bt_tuples.append(("Wolfgang Clement", "SPD", '14'))
bt_tuples.append(("Christina Weiss", "fraktionslos", '14'))
bt_tuples.append(("Julian Nida-Rümelin", "SPD", '14'))
bt_tuples.append(("Michael Naumann", "SPD", '14'))
bt_tuples.append(("Werner Tegtmeier", "SPD", '13'))
bt_tuples.append(("Jürgen Stark", "fraktionslos", '13'))
bt_tuples.append(("Hans-Friedrich von Ploetz", "fraktionslos", '13'))
bt_tuples.append(("Baldur Wagner", "CDU/CSU", '12'))
bt_tuples.append(("Karl Jung", "fraktionslos", '12'))
bt_tuples.append(("Franz-Josef Feiter", "fraktionslos", '12'))
bt_tuples.append(("Manfred Overhaus", "fraktionslos", '12'))
bt_tuples.append(("Wighard Härdtl", "CDU/CSU", '12'))
bt_tuples.append(("Wilhelm Knittel", "CDU/CSU", '12'))
bt_tuples.append(("Clemens Stroetmann", "CDU/CSU", '12'))
bt_tuples.append(("Franz Kroppenstedt", "fraktionslos", '12'))
bt_tuples.append(("Hans-Joachim Fuchtel", "CDU/CSU", '12'))
bt_tuples.append(("Frerich Görts", "CDU/CSU", '12'))

## Doppelt vorkommende Namen

Damit die Reden von Ministern, Staatssekretären etc. auch eindeutig einer Partei zugeordnet werden können, ist es wichtig, dass nicht zu viele doppelte Namens-Einträge in der Liste enthalten sind.  
In diesem Codeblock werden deswegen doppelt vorhandene Namen gefunden und ausgegeben.

In [5]:
def find_duplicates_by_name(list_of_tuples):
    name_map = {}
    duplicates = []
    
    for tup in list_of_tuples:
        name = tup[0]
        if name in name_map:
            name_map[name].append(tup)
            if len(name_map[name]) == 2:
                duplicates.extend(name_map[name])
        else:
            name_map[name] = [tup]
    
    return duplicates
dupes = find_duplicates_by_name(bt_tuples)
print(len(dupes))
dupes

32


[('Günter Klein', 'SPD', '4'),
 ('Günter Klein', 'CDU', '12'),
 ('Heinrich Müller', 'SPD', '1'),
 ('Heinrich Müller', 'SPD', '8'),
 ('Karl Müller', 'CDU', '2'),
 ('Karl Müller', 'SPD', '5'),
 ('Rudolf Müller', 'CDU', '6'),
 ('Rudolf Müller', 'SPD', '12'),
 ('Christian Schmidt', 'GRÜNE', '10'),
 ('Christian Schmidt', 'CSU', '19'),
 ('Manfred Schmidt', 'SPD', '11'),
 ('Manfred Schmidt', 'CDU', '8'),
 ('Wilhelm Schmidt', 'WAV', '1'),
 ('Wilhelm Schmidt', 'SPD', '15'),
 ('Gerhard Schröder', 'CDU', '8'),
 ('Gerhard Schröder', 'SPD', '16'),
 ('Karl Weber', 'CDU', '4'),
 ('Karl Weber', 'CDU', '8'),
 ('Frank Schmidt', 'CDU', '11'),
 ('Frank Schmidt', 'SPD', '16'),
 ('Peter Friedrich', 'SPD', '14'),
 ('Peter Friedrich', 'SPD', '17'),
 ('Harald Koch', 'SPD', '1'),
 ('Harald Koch', 'DIE LINKE', '17'),
 ('Alois Rainer', 'CSU', '9'),
 ('Alois Rainer', 'CSU', '20'),
 ('Dagmar Schmidt', 'SPD', '16'),
 ('Dagmar Schmidt', 'SPD', '20'),
 ('Michael Müller', 'SPD', '16'),
 ('Michael Müller', 'SPD', '20'),

**Zusammenfassend**

In der Liste sind relativ wenige doppelte Einträge. Immerhin ist die Liste über 4000 Einträge lang.  
Da die Liste nur bei Ministern, Kanzlern und Staatssekretären zur Ermittlung der Parteien genutzt wird, sollten die wenigen doppelten Namen keinen merkbaren Einfluss auf die Endergebnisse der Analysen haben (wenn sie sich überhaupt darauf auswirken).

### Eine Methode, um die Partei für einen bestimmten Namen herauszufinden könnte dann so aussehen 

Das wird im nächsten Notebook wichtig.

In [6]:
import re
def find_party_by_name(name, tuple_list):
    # sort the list in descending order by Bundestag period. This ensures that newer entries are preferred in cases of duplicate entries.
    tuple_list_sorted = sorted(tuple_list, key = lambda x: x[2], reverse = True) 
    if name.startswith('Dr.'):
        name = name[4:]
    for item in tuple_list_sorted:
        if name in item[0]:
            if item[1] != 'CSU' and item[1] != 'CDU':
                return item[1]
            else:
                return 'CDU/CSU'  
    return "<unknown>"

print(find_party_by_name ('Dr. Norbert Blüm', bt_tuples))
print(find_party_by_name ('Ursula Seiler-Albring', bt_tuples))

CDU/CSU
FDP


In [7]:
bt_tuples = sorted(bt_tuples, key = lambda x: int(x[2]), reverse = True)

Speichern in einer CSV-Datei, damit wir in den nächsten Notebooks leichten Zugriff darauf haben.

In [8]:
import csv

with open('../_data/abgeordnete.csv','w', encoding='utf8', newline='') as out:
    csv_out=csv.writer(out)
    csv_out.writerow(['name','party', 'BT-Period'])
    csv_out.writerows(bt_tuples) 
